## Data curation experiment
Idea: the kernel of the surrogate model represents the relationship between outcomes across the factor space -> it helps us understand how the mean outcome across the factor space could change if one or more outcomes were improved (i.e. collected demos for)

Terms:
- $y_{expert}$: the maximum outcome achievable
- $\mu(x)$: expected outcome at factor configuration $x$

For a single candidate demo $x^*$, the expected new mean outcome at any other point $x$ in the space is:
$$\mu_{new}(x) = \mu_{current}(x) + \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} (y_{expert} - \mu_{current}(x^*))$$

To calculate the expected change in mean outcome across the entire factor space, we integrate that change over the whole domain $\mathcal{X}$.
$$\text{Total Influence}(x^*) = \left( y_{expert} - \mu_{current}(x^*) \right) \cdot \int_{\mathcal{X}} \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} dx$$

To find the $x^*$ that maximizes the overall average change, simply calculate this score for all candidates and select the highest one.

What if you wanted to select more than one point (a batch $X_{batch} = [x_1, x_2, \dots, x_N]$) to collect demos for?
To score a full batch, you upgrade the scalar covariance $k(x, x^*)$ into a covariance matrix $K(X_{batch}, X_{batch})$. The expected total influence for a batch becomes:$$\text{Total Influence}(X_{batch}) = \int_{\mathcal{X}} K(x, X_{batch}) \left[ K(X_{batch}, X_{batch}) + \Sigma_{noise} \right]^{-1} (Y_{expert} - M_{current}(X_{batch})) dx$$

In [1]:
import torch
import pickle
import pandas as pd
from factors_config import FACTOR_COLUMNS, get_design_points_robot, BOUNDS, tkwargs

In [2]:
TASKS = ['pickblueblock', 'uprightcup', 'putgreeninpot']
MAX_OUTCOME = {'pickblueblock': 2.0, 'uprightcup': 3.0, 'putgreeninpot': 6.0}

In [11]:
# Load active model
active_model_path = './results/putgreeninpot_active_offline_SingleTaskGP_PSD/run_1/models/trial_100_model.pkl'

with open(active_model_path, 'rb') as f:
    active_model_data = pickle.load(f)
    active_model = active_model_data['model']
    active_model_name = active_model_data.get('model_name', 'Unknown')
print(f"Loaded active model: {active_model_name}")

Loaded active model: SingleTaskGP


In [12]:
# See covariance matrix on training data
bf_data = pd.read_csv('./results/putgreeninpot_bruteforce/results.csv')
X_all = torch.tensor(bf_data[FACTOR_COLUMNS].values, **tkwargs)
y_all = torch.tensor(bf_data['continuous_outcome'].values, **tkwargs).unsqueeze(-1)

active_model.covar_module(X_all).to_dense()

tensor([[1.0000, 0.9895, 0.9587,  ..., 0.0000, 0.0000, 0.0000],
        [0.9895, 1.0000, 0.9895,  ..., 0.0000, 0.0000, 0.0000],
        [0.9587, 0.9895, 1.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 1.0000, 0.9895, 0.9587],
        [0.0000, 0.0000, 0.0000,  ..., 0.9895, 1.0000, 0.9895],
        [0.0000, 0.0000, 0.0000,  ..., 0.9587, 0.9895, 1.0000]],
       dtype=torch.float64, grad_fn=<ExpBackward0>)

### Calculating batch total influence + greedy and stochastic greedy batch search

In [13]:
def calculate_batch_total_influence(X_batch, active_model, X_all, y_target):
    """
    Scores a proposed batch of points based on expected total influence.
    """
    # 1. Ensure model is in evaluation mode
    active_model.eval()
    active_model.likelihood.eval()
    
    with torch.no_grad():
        # 2. Get the current mean prediction for the batch; this is what we're trying to improve
        # shape: (B, 1)
        mean_batch = active_model.posterior(X_batch).mean
        
        # 3. Calculate the batch covariance matrix K(X_batch, X_batch)
        # shape: (B, B)
        K_bb = active_model.covar_module(X_batch).to_dense()
        
        # 4. Add the model's observation noise to the diagonal
        noise = active_model.likelihood.noise.item() # scalar, assuming homoskedastic noise (default for SingleTaskGP)
        K_bb_noisy = K_bb + noise * torch.eye(X_batch.size(0), device=X_batch.device, dtype=X_batch.dtype)
        
        # 5. Calculate the cross-covariance between the whole space and the batch K(X_all, X_batch)
        # shape: (N_all, B)
        K_ab = active_model.covar_module(X_all, X_batch).to_dense()
        
        # 6. Calculate the difference from the target outcome
        # shape: (B, 1)
        y_diff = y_target - mean_batch
        
        # 7. Find the product: [K_bb_noisy]^-1 * y_diff
        # We use torch.linalg.solve instead of explicit inverse for numerical stability
        # Essentially saying K_bb_noisy = A, y_diff = b, inv_term = x for Ax=b 
        # shape: (B, 1)
        inv_term = torch.linalg.solve(K_bb_noisy, y_diff)
        
        # 8. Multiply by cross-covariance and sum over the entire design space X_all
        # shape: (N_all, 1) -> sum to a single scalar
        influence_per_point = torch.matmul(K_ab, inv_term)
        total_influence = influence_per_point.sum()
        
    return total_influence.item()

In [25]:
# Greedy batch selection
def propose_batch_greedy(batch_size, active_model, X_all, y_target):
    """
    Greedily builds a batch of size `batch_size` that maximizes total influence.
    """    
    # Track the indices of points we've selected from X_all
    selected_indices = []
    
    print(f"Proposing a batch of {batch_size} points...")
    
    for step in range(batch_size):
        best_score = -float('inf')
        best_idx = -1
        
        # Evaluate adding each possible point to our current batch
        for i in range(len(X_all)):
            if i in selected_indices:
                continue
                
            # Create a temporary batch with the candidate point
            current_batch_indices = selected_indices + [i]
            X_batch_candidate = X_all[current_batch_indices]
            
            # Score the temporary batch
            score = calculate_batch_total_influence(X_batch_candidate, active_model, X_all, y_target)
            
            if score > best_score:
                best_score = score
                best_idx = i
                
        selected_indices.append(best_idx)
        print(f"  Step {step+1}/{batch_size}: Added point index {best_idx} | New Batch Influence Score: {best_score:.4f}")
        
    X_proposed = X_all[selected_indices]
    return X_proposed, selected_indices, best_score

In [35]:
# Stochastic greedy batch selection
import random

def propose_batch_local_search(batch_size, active_model, X_all, y_target, max_iters=1000):
    """
    Improves a batch using Stochastic Local Search (Swapping) over a discrete grid.
    """
    
    # 1. Initialize with the Greedy approach for a strong head start
    print("Initializing with Greedy search...")
    _, current_indices, best_score = propose_batch_greedy(batch_size, active_model, X_all, y_target)
    
    print(f"\nStarting Local Search Optimization. Initial Score: {best_score:.4f}")
    
    # 2. Iteratively attempt to swap points to find synergistic combinations
    improvements = 0
    all_indices = set(range(len(X_all)))
    
    for iteration in range(max_iters):
        # Pick a random point currently IN the batch to remove
        idx_to_remove = random.choice(current_indices)
        
        # Pick a random point currently OUTSIDE the batch to add
        available_indices = list(all_indices - set(current_indices))
        idx_to_add = random.choice(available_indices)
        
        # Create the proposed new batch indices
        proposed_indices = current_indices.copy()
        proposed_indices.remove(idx_to_remove)
        proposed_indices.append(idx_to_add)
        
        # Evaluate the proposed batch
        proposed_X_batch = X_all[proposed_indices]
        proposed_score = calculate_batch_total_influence(proposed_X_batch, active_model, X_all, y_target)
        
        # 3. If the swap improved the joint score, keep it!
        if proposed_score > best_score:
            current_indices = proposed_indices
            best_score = proposed_score
            improvements += 1
            print(f"  [Iter {iteration}] Improved! Swapped {idx_to_remove} for {idx_to_add}. New Score: {best_score:.4f}")
            
    print(f"\nLocal Search Complete. Made {improvements} improvements.")
    
    final_X_batch = X_all[current_indices]
    return final_X_batch, current_indices, best_score

### Test these selection strategies out

In [16]:
# Define your target/max outcome (e.g. 3.0 for the 'uprightcup' task)
Y_TARGET = 6.0  # change for different tasks
BATCH_SIZE = 20  # how many points to collect demos at

In [29]:
# --- Greedy selection ---
greedy_X, greedy_indices, greedy_score = propose_batch_greedy(
    batch_size=BATCH_SIZE,
    active_model=active_model,
    X_all=X_all,
    y_target=Y_TARGET
)

print(f"Greedy Score: {greedy_score:.4f}")
# Show proposed points in a table
# print("\nProposed Factor Configurations to Collect Next:")
# print(pd.DataFrame(data=proposed_X.numpy(), columns=FACTOR_COLUMNS))

Proposing a batch of 20 points...
  Step 1/20: Added point index 44 | New Batch Influence Score: 271.8773
  Step 2/20: Added point index 116 | New Batch Influence Score: 506.3108
  Step 3/20: Added point index 278 | New Batch Influence Score: 730.8494
  Step 4/20: Added point index 612 | New Batch Influence Score: 939.7510
  Step 5/20: Added point index 429 | New Batch Influence Score: 1110.2074
  Step 6/20: Added point index 348 | New Batch Influence Score: 1257.9781
  Step 7/20: Added point index 166 | New Batch Influence Score: 1397.4613
  Step 8/20: Added point index 534 | New Batch Influence Score: 1528.5448
  Step 9/20: Added point index 649 | New Batch Influence Score: 1653.2158
  Step 10/20: Added point index 13 | New Batch Influence Score: 1695.2653
  Step 11/20: Added point index 67 | New Batch Influence Score: 1732.6251
  Step 12/20: Added point index 142 | New Batch Influence Score: 1768.2123
  Step 13/20: Added point index 293 | New Batch Influence Score: 1803.0955
  Step 

In [36]:
# --- Stochastic greedy selection ---
stoch_greedy_X, stoch_greedy_indices, stoch_greedy_score = propose_batch_local_search(
    batch_size=BATCH_SIZE, 
    active_model=active_model, 
    X_all=X_all, 
    y_target=Y_TARGET, 
    max_iters=1000  # Adjust based on your compute budget
)

print(f"Stochastic Greedy Score: {stoch_greedy_score:.4f}")
# print("\nProposed Factor Configurations to Collect Next:")
# print(pd.DataFrame(data=optimized_X.numpy(), columns=FACTOR_COLUMNS))

Initializing with Greedy search...
Proposing a batch of 20 points...
  Step 1/20: Added point index 44 | New Batch Influence Score: 271.8773
  Step 2/20: Added point index 116 | New Batch Influence Score: 506.3108
  Step 3/20: Added point index 278 | New Batch Influence Score: 730.8494
  Step 4/20: Added point index 612 | New Batch Influence Score: 939.7510
  Step 5/20: Added point index 429 | New Batch Influence Score: 1110.2074
  Step 6/20: Added point index 348 | New Batch Influence Score: 1257.9781
  Step 7/20: Added point index 166 | New Batch Influence Score: 1397.4613
  Step 8/20: Added point index 534 | New Batch Influence Score: 1528.5448
  Step 9/20: Added point index 649 | New Batch Influence Score: 1653.2158
  Step 10/20: Added point index 13 | New Batch Influence Score: 1695.2653
  Step 11/20: Added point index 67 | New Batch Influence Score: 1732.6251
  Step 12/20: Added point index 142 | New Batch Influence Score: 1768.2123
  Step 13/20: Added point index 293 | New Batch

In [41]:
# Compare to "certain failures" selection
certain_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_certain_failures.csv')
print(f"Loaded {len(certain_failures_points)} points")
cf_X = torch.tensor(certain_failures_points[FACTOR_COLUMNS].values, **tkwargs)
cf_score = calculate_batch_total_influence(cf_X, active_model, X_all, Y_TARGET)
print(f"Certain Failures Score: {cf_score:.4f}")

Loaded 20 points
Certain Failures Score: 939.0365


In [40]:
# Compare to "observed failures" selection
observed_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_observed_failures.csv')
print(f"Loaded {len(observed_failures_points)} points")
of_X = torch.tensor(observed_failures_points[FACTOR_COLUMNS].values, **tkwargs)
of_score = calculate_batch_total_influence(of_X, active_model, X_all, Y_TARGET)
print(f"Observed Failures Score: {of_score:.4f}")

Loaded 20 points
Observed Failures Score: 1063.7690


In [24]:
# Compare to random selection
random_scores = []
# score n random batches, take the average
for i in range(10):
    random_indices = torch.randperm(len(X_all))[:BATCH_SIZE]
    random_X_batch = X_all[random_indices]
    random_score = calculate_batch_total_influence(random_X_batch, active_model, X_all, Y_TARGET)
    random_scores.append(random_score)

random_score = sum(random_scores) / len(random_scores)
print(f"Random Selection Score: {random_score:.4f}")

Random Selection Score: 1475.5175


In [38]:
# Compare all influence scores
print("--- Influence Scores ---")
print(f"Greedy: {greedy_score:.4f}")
print(f"Stochastic Greedy: {stoch_greedy_score:.4f}")
print(f"Certain Failures: {cf_score:.4f}")
print(f"Observed Failures: {of_score:.4f}")
print(f"Random Selection: {random_score:.4f}")

--- Influence Scores ---
Greedy: 2005.3174
Stochastic Greedy: 2033.9239
Certain Failures: 939.0365
Observed Failures: 1063.7690
Random Selection: 1475.5175
